In [1]:
!git clone https://github.com/Vedant988/tmz-R2.git

Cloning into 'tmz-R2'...
remote: Enumerating objects: 44, done.
remote: Counting objects: 100% (44/44), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 44 (delta 13), reused 36 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (44/44), 6.78 KiB | 6.78 MiB/s, done.
Resolving deltas: 100% (13/13), done.


In [1]:
%cd /content/tmz-R2
!pip install -r requirements.txt

/content/tmz-R2
  Cloning https://github.com/VarunGumma/IndicTransToolkit.git to /tmp/pip-install-tfhojp61/indictranstoolkit_69b3811ec655476d98b70c9e4f82070c
  Running command git clone --filter=blob:none --quiet https://github.com/VarunGumma/IndicTransToolkit.git /tmp/pip-install-tfhojp61/indictranstoolkit_69b3811ec655476d98b70c9e4f82070c
  Resolved https://github.com/VarunGumma/IndicTransToolkit.git to commit 3efb8418d0721b4ce267c2b3586899d313191357
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [10]:
# %cd /content/tmz-R2
# !git pull

/content/tmz-R2
remote: Enumerating objects: 8, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 5 (delta 3), reused 5 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 659 bytes | 659.00 KiB/s, done.
From https://github.com/Vedant988/tmz-R2
   9934fcd..fa76ba1  main       -> origin/main
Updating 9934fcd..fa76ba1
Fast-forward
 requirements.txt                           | 4 +++-
 task2_asr_transliteration/requirements.txt | 2 +-
 2 files changed, 4 insertions(+), 2 deletions(-)


# cell 1

In [2]:
import pandas as pd
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from tqdm import tqdm
import sacrebleu
from google.colab import userdata
import os

# Authenticate using your Colab Secret
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    print("HF_TOKEN secret not found. Make sure you added it to the Secrets sidebar.")

# Device configuration (Uses T4 GPU)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load IndicTrans2 (1B Parameter Model)
model_name = "ai4bharat/indictrans2-en-indic-1B"

# trust_remote_code=True is required for AI4Bharat's custom tokenization
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

# We use torch.float16 for faster inference and lower VRAM usage on T4
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    trust_remote_code=True,
    torch_dtype=torch.float16
).to(device)

print("IndicTrans2-1B loaded successfully!")

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenization_indictrans.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-en-indic-1B:
- tokenization_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

modeling_indictrans.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-en-indic-1B:
- modeling_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


model.safetensors:   0%|          | 0.00/4.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

IndicTrans2-1B loaded successfully!


# Cell 2

In [3]:
from IndicTransToolkit.processor import IndicProcessor

# Initialize the processor
ip = IndicProcessor(inference=True)

def translate_en_to_ta(text):
    src_lang = "eng_Latn"
    tgt_lang = "tam_Taml"

    # 1. Preprocess the text (this normalizes the text and safely prepends the language tags)
    processed_text = ip.preprocess_batch([text], src_lang=src_lang, tgt_lang=tgt_lang)

    # 2. Tokenize
    inputs = tokenizer(
        processed_text,
        truncation=True,
        padding="longest",
        return_tensors="pt",
        return_attention_mask=True
    ).to(device)

    # 3. Generate translation (No need for forced_bos_token_id anymore!)
    with torch.inference_mode():
        translated_tokens = model.generate(
            **inputs,
            max_length=256,
            num_beams=5
        )

    # 4. Decode
    decoded_text = tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)

    # 5. Postprocess (cleans up any remaining artifacts for a clean BLEU score)
    return ip.postprocess_batch(decoded_text, lang=tgt_lang)[0]

# Test translation
sample = "Hello, how are you? Welcome to the Indic Translation project."
print("Sample Translation:", translate_en_to_ta(sample))

Sample Translation: வணக்கம், எப்படி இருக்கிறீர்கள்? இந்திய மொழிபெயர்ப்பு திட்டத்திற்கு வரவேற்கிறோம்.


# Cell 3

In [6]:
data = {
    'english_source': ["This is a test sentence.", "Machine learning is fascinating.", "I love programming."],
    'tamil_reference': ["இது ஒரு சோதனை வாக்கியம்.", "இயந்திர கற்றல் கவர்ச்சிகரமானது.", "எனக்கு நிரலாக்கம் பிடிக்கும்."]
}
df = pd.DataFrame(data)

df = pd.read_csv('data/raw/translation_dataset.csv')

tqdm.pandas(desc="Translating")
df['indictrans2_prediction'] = df['english_source'].progress_apply(translate_en_to_ta)

# Evaluate using SacreBLEU
bleu_score = sacrebleu.corpus_bleu(
    df['indictrans2_prediction'].tolist(),
    [df['tamil_reference'].tolist()]
)

print(f"SacreBLEU Score: {bleu_score.score:.2f}")

display(df[['english_source', 'indictrans2_prediction', 'tamil_reference']])

# Save the outputs according to your structure
df.to_csv('task1_translation_evaluation/part_a_batch_translation/translation_outputs.csv', index=False)


Translating: 100%|██████████| 15/15 [00:04<00:00,  3.60it/s]

SacreBLEU Score: 42.49


,english_source,indictrans2_prediction,tamil_reference
0,This is a test sentence.,இது ஒரு சோதனை வாக்கியம்.,இது ஒரு சோதனை வாக்கியம்.
1,Machine learning is fascinating.,இயந்திர கற்றல் சுவாரஸ்யமானது.,இயந்திர கற்றல் கவர்ச்சிகரமானது.
2,I love programming.,எனக்கு புரோகிராமிங் பிடிக்கும்.,எனக்கு நிரலாக்கம் பிடிக்கும்.
3,Artificial intelligence is transforming the wo...,செயற்கை நுண்ணறிவு உலகை மாற்றியமைத்து வருகிறது.,செயற்கை நுண்ணறிவு உலகத்தை மாற்றிக் கொண்டிருக்க...
4,The weather is beautiful today.,இன்று வானிலை அழகாக உள்ளது.,இன்று வானிலை அழகாக இருக்கிறது.
5,Please let me know if you need any help.,உங்களுக்கு ஏதேனும் உதவி தேவைப்பட்டால் எனக்கு த...,உங்களுக்கு ஏதாவது உதவி தேவைப்பட்டால் தயவுசெய்த...
6,He is reading a book in the library.,நூலகத்தில் ஒரு புத்தகத்தைப் படிக்கிறார்.,அவர் நூலகத்தில் ஒரு புத்தகம் படித்துக் கொண்டிர...
7,We are going to the market to buy vegetables.,காய்கறிகள் வாங்க சந்தைக்குச் செல்கிறோம்.,நாங்கள் காய்கறிகள் வாங்க சந்தைக்குச் செல்கிறோம்.
8,The quick brown fox jumps over the lazy dog.,வேகமாக பழுப்பு நிற நரி சோம்பேறி நாய் மீது குதி...,வேகமான பழுப்பு நரி சோம்பேறி நாயின் மீது குதிக்...
9,Data structures and algorithms are essential f...,மென்பொருள் பொறியாளர்களுக்கு தரவு கட்டமைப்புகள்...,மென்பொருள் பொறியாளர்களுக்கு தரவு கட்டமைப்புகள்...


In [8]:
# Save SacreBLEU score to match the required repo structure
bleu_df = pd.DataFrame({'metric': ['SacreBLEU'], 'score': [bleu_score.score]})
bleu_df.to_csv('task1_translation_evaluation/part_a_batch_translation/sacrebleu_results.csv', index=False)
print(" sacrebleu_results.csv saved!")

 sacrebleu_results.csv saved!


---